In [26]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

TRAIN_PATH = "Dataset/train"
TEST_PATH = "Dataset/test"
MODEL_PATH = "Model/model.xml"

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

In [27]:
def preprocess(img):
    img = cv2.resize(img, (100, 100))
    img = cv2.equalizeHist(img)
    return img

def detect_face(img):
    faces = face_cascade.detectMultiScale(img, 1.05, 2, minSize=(30,30))
    if len(faces) > 0:
        x,y,w,h = faces[0]
        return preprocess(img[y:y+h, x:x+w]), (x,y,w,h)
    
    w, h = img.shape
    return preprocess(img), (0,0,w,h)

In [28]:
def load_data(folder_path, class_names):
    faces, labels = [], []
    for label, class_name in enumerate(class_names):
        class_path = os.path.join(folder_path, class_name)
        if not os.path.isdir(class_path):
            continue
        for filename in sorted(os.listdir(class_path)):
            img = cv2.imread(os.path.join(class_path, filename), cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue
            face_img, _ = detect_face(img)
            faces.append(face_img)
            labels.append(label)

    return faces, np.array(labels, dtype=np.uint32)

In [29]:
def train_test_model():
    class_names = sorted(os.listdir(TRAIN_PATH))

    print("Train Data Loading...")
    train_faces, label_faces = load_data(TRAIN_PATH, class_names)
    if not train_faces:
        print("Training Data Not Found!")
        return

    face_recognizer = cv2.face.LBPHFaceRecognizer_create()
    face_recognizer.train(train_faces, label_faces)
    
    print("Test Data Loading...")
    test_faces, label_faces = load_data(TEST_PATH, class_names)
    if not test_faces:
        print("Test Data Not Found!")
        return
    
    correct = 0
    for face, true_label in zip(test_faces, label_faces):
        predicted_label, confidence = face_recognizer.predict(face)
        status = "RIGHT" if predicted_label == true_label else "WRONG"
        correct += predicted_label == true_label
        print(f"Actual: {class_names[true_label]} | Predicted: {class_names[predicted_label]} | Confidence: {confidence:.2f} | {status}")

    print(f"Average Accuracy: {correct/len(test_faces) * 100:2f}%")

    os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)
    face_recognizer.write(MODEL_PATH)
    print("Model Saved Successfully!")

In [ ]:
def predict(img_path):
    if not os.path.exists(MODEL_PATH):
        print("Model Not Found! Train Model First")
        return

    class_names = sorted(os.listdir(TRAIN_PATH))
    face_recognizer = cv2.face.LBPHFaceRecognizer_create()
    face_recognizer.read(MODEL_PATH)

    img = cv2.imread(img_path)
    if img is None:
        print("Error! Cant Read Image")
        return
    
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    face_img, (x,y,w,h) = detect_face(img_gray)
    predicted_label, confidence = face_recognizer.predict(face_img)
    predicted_class = class_names[predicted_label] 

    cv2.rectangle(img, (x, y), (x+w, y+h), (255,0,0), 2)
    cv2.putText(img, f"{predicted_class} | {confidence:.2f}", (x, y-10), cv2.FONT_HERSHEY_PLAIN, 1, (0,255,0))

    print(f"{img_path} detected as {predicted_class}")
    print(f"Detected Subject    : {predicted_class}")
    print(f"Confidence          : {confidence:2f}")
    print(f"Detected Face Loc   : x={x}, y={y}, w={w}, h={h}")

    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()




In [31]:
def main():
    while True:
        print("\n===FACE DETECTION===")
        print("1. Train and Test Data")
        print("2. Predict")
        print("3. Exit")
        choice = input("Input your choice (1-3): ")
        if choice == '1':
            train_test_model()
        elif choice == '2':
            img_path = input("Choose your image path for prediction: ")
            predict(img_path)
        elif choice == '3':
            print("Exit...")
            break
        else:
            print("Invalid Input. Please Choice 1-3!")

main()


===FACE DETECTION===
1. Train and Test Data
2. Predict
3. Exit
Train Data Loading...


error: OpenCV(4.13.0) :-1: error: (-5:Bad argument) in function 'train'
> Overload resolution failed:
>  - labels data type = uint32 is not supported
>  - Expected Ptr<cv::UMat> for argument 'labels'
